> Check the signals in some clusters

In [ ]:
import pandas as pd

import plotly
import plotly.express as px
import plotly.graph_objects as go

import trapallada.analysis

from IPython.display import display, Markdown

So that `import`ed Python modules are automatically reloaded when modified

In [ ]:
%load_ext autoreload
%autoreload 2

So that *plotly* pictures can be exported to HTML

In [ ]:
plotly.offline.init_notebook_mode()

# Input Files

Relative path and *Base* file name for the results (trained model). It should be a `str`.

In [ ]:
# input_files_stem = '../results/epochs/20000/4_sources/epochs'
# input_files_stem = '../results/4_sources/vanilla_5000_epochs_05_x_sd'
# ===============
# input_files_stem = '../results/new/4_sources/vanilla'
# input_files_stem = '../results/new/4_sources/fft_vanilla'
input_files_stem = '../results/new/4_sources/fft_vanilla_x_sd_2'

In [ ]:
ds, positions_df, model, results_df, parameters, analysis_df = trapallada.analysis.load_results_dir(input_files_stem, analysis_also=True)

In [ ]:
analysis_df.head()

For the sake of convenience

In [ ]:
# pos_columns = ['RA', 'DEC', 'CRA', 'CDEC']
tsne_columns = [f'tsne_{i}' for i in range(1, 3)]

The number of objects for every *type*

In [ ]:
analysis_df['main_type'].value_counts()

The clustering (either *in projected space* or *in z*) to be analyzed

In [ ]:
cluster = 'cluster in projected space'
# cluster = 'cluster in z'

A few `main_type`s are selected

In [ ]:
on_plot_hover_columns = ['index', 'source', cluster, 'main_type']
on_plot_hover_columns

In [ ]:
for c in ['cluster in projected space', 'cluster in z']:
    analysis_df[cluster] = pd.Categorical(analysis_df[cluster])
    # display(analysis_df[c].value_counts())

# All the signals

Empty `main_type`s are set to `UNKNOWN`

In [ ]:
analysis_df['main_type'].fillna('UNKNOWN', inplace=True)

In order to have fixed axes

In [ ]:
# analysis_df[tsne_columns].min(), analysis_df[tsne_columns].max()
xrange, yrange = list(zip(analysis_df[tsne_columns].min(), analysis_df[tsne_columns].max()))
xrange, yrange

One can **double-click** a `main_type` in the *Legend* to only show the corresponding curves

In [ ]:
analysis_df['main_type'].unique().shape

In [ ]:
fig = px.scatter(
    analysis_df.reset_index(),
    *tsne_columns,
    color='main_type',
    hover_data=on_plot_hover_columns, opacity=0.5,
    width=1_200, height=800)
fig.show()

A convenience function

In [ ]:
def mark_rectangle(fig, x_range: list[float], y_range: list[float], text: str, n_samples: int = 5):

    # the rectangle is overimposed on the picture
    fig.add_shape(
        type='rect',
        x0=x_range[0], y1=y_range[0], 
        x1=x_range[1], y0=y_range[1],
        line=dict(color='RoyalBlue'),
    )

    fig.add_annotation(
        x=x_range[1], y=y_range[1],
        text=text,
        showarrow=False,
        # arrowhead=1
    )

    return fig

In [ ]:
def sample_and_plot_range(n: int, x_range, y_range):

    # `True` if the tSNE coordinates of the corresponding point are inside the rectangle
    belong = (analysis_df['tsne_1'] > x_range[0]) &\
        (analysis_df['tsne_1'] < x_range[1]) &\
        (analysis_df['tsne_2'] < y_range[1]) &\
        (analysis_df['tsne_2'] > y_range[0])

    # the original `DataFrame` is subset
    selected_df = analysis_df[belong]

    n = min(n, len(selected_df))

    for i in selected_df.sample(n, random_state=42).index:
        
        display(Markdown(f'### {i}'))
        
        print(analysis_df.filter(regex='^(?!z_|tsne_|set$)').loc[i])
        
        display(trapallada.analysis.predict_and_plot(model, ds, i, display_metadata=False))

In [ ]:
def inspect_regions(fig, n_samples, x_range_list, y_range_list, labels):
    
    for x_range, y_range, label in zip(x_range_list, y_range_list, labels):
        
        fig = mark_rectangle(fig, x_range, y_range, label)

    display(Markdown('## Regions'))
    
    fig.show()

    zoomed_in_fig = go.Figure(fig)

    for x_range, y_range, label in zip(x_range_list, y_range_list, labels):

        display(Markdown(f'### {label}'))

        zoomed_in_fig.update_layout(xaxis=dict(range=x_range),yaxis=dict(range=y_range))

        zoomed_in_fig.show()

        sample_and_plot_range(n_samples, x_range, y_range)

# Zooming in

In [ ]:
# regions = [
#     {
#         'x_range': (-30., 6.5),
#         'y_range': (-78., -53.),
#     },
#     {
#         'x_range': (33.8, 56.8),
#         'y_range': (32., 53.5),
#     },
#     {
#         'x_range': (17.8, 28.),
#         'y_range': (55.8, 65.5),
#     },
# ]
regions = [
    {
        'x_range': (-5., 20.),
        'y_range': (100., 125.),
    },
    {
        'x_range': (106, 120),
        'y_range': (-35., -15),
    },
    {
        'x_range': (-20, 20),
        'y_range': (-20, 20),
    },
]

In [ ]:
x_range_list = list(map(lambda x: x['x_range'], regions))
y_range_list = list(map(lambda x: x['y_range'], regions))
labels = [f'Region {i}' for i in range(len(x_range_list))]

inspect_regions(go.Figure(fig), 5, x_range_list, y_range_list, labels)